In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
import pandas as pd
import re
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from chunking_evaluation.chunking import KamradtModifiedChunker
from sentence_transformers import SentenceTransformer
from chunking_evaluation.chunking import ClusterSemanticChunker
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
print(torch.version.cuda)

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


12.1


In [2]:
train = pd.read_csv('data/train.csv')

In [3]:
validation = train.iloc[700:]
train = train[:700]

In [4]:
train.head()

,paper_id,text,summary
0,0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [5]:
validation.shape

(300, 3)

In [5]:
model_name = 'google/flan-t5-large'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 937.58it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
indices = [10]

for i in indices:
    text = train['text'][i]
    summary = train['summary'][i]

    prompt = f"""
Summarize the following text using around 200 words at least {text}

Summary:
"""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    input = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(input['input_ids'], max_new_tokens=50,)[0]
    )

    print(summary)
    print("--------------------")
    print(output)



Token indices sequence length is longer than the specified maximum sequence length for this model (5852 > 512). Running this sequence through the model will result in indexing errors


OBJECTIVES: Evidence on how individual characteristics and distancing policies during the first wave of COVID-19 together influenced health behaviours is scarce. The objective of this study is to fill in this gap by studying how the propensity to engage in protective behaviours in Europe was shaped by the interplay of individual characteristics and national policies. 
DESIGN: Data on individual behaviour in 27 countries came from the “Corona Survey” module of the Survey of Health, Ageing and Retirement in Europe, collected in summer 2020. As outcomes, we considered avoidant behaviours (never leaving home, reducing frequency of walks, reducing frequency of social meetings) and preventive behaviour (wearing a face mask). Among relevant policies we considered stay-at-home restrictions, mask wearing policies, and gatherings’ restrictions. Individual characteristics comprised gender, health risk of COVID-19 (older age and poor health), and activity (employment and providing help to other ho

In [8]:
indices = [10, 45, 38]
index = enumerate(indices)

In [9]:
text = train['text'][3]
paragraphs = re.split("\n\n", text)

In [10]:
split = text.split()
[' '.join(split[i:i+500]) for i in range(0, len(split), 500)]

["## 1 Problem Setup Recent political science scholarship has questioned the ability to evaluate presidential election forecasts [1]. On its face, this argument is plausible: presidential elections are rare, and state-level outcomes within presidential elections are highly correlated. If we believe we need 10 or 20 or 100 presidential elections to evaluate whether a forecast provides useful information, we could be waiting a long time wondering if we are being lead astray by forecasts or not. Despite the seeming plausibility of the argument by Grimmer, Knox, and Westwood (henceforth GKW), I argue 1 that there are several ways we may evaluate election forecasts on shorter timescales than 'decades to millennia.' As a heuristic, I argue that we can generally determine whether a forecast is better than random guessing using only publicly available data no more granular than the congressional district level on timescales of two-to-three election cycles (4-6 years in the U.S. counting both m

In [8]:
def cleaning_text(text):
     # quitar referencias tipo [1]
    text = re.sub(r"\[\d+\]", "", text)
    # limpiar caracteres
    text = re.sub(r"[^a-zA-Z0-9\s.,%\-]", " ", text)
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
train['clean_text'] = train['text'].apply(cleaning_text)
validation['clean_text'] = validation['text'].apply(cleaning_text)

In [12]:
def simple_split(text, chunk_size=60):
    txt_split = text.split()
    return [' '.join(txt_split[i:i+chunk_size]) for i in range(0, len(txt_split), chunk_size)]

In [10]:
sentence_transf_model = SentenceTransformer("BAAI/bge-base-en", device='cuda')
def embedding_function(texts):
    return sentence_transf_model.encode(texts, batch_size=64, show_progress_bar=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1010.09it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
all_chunks = []
chunk_pos = []

for i, text in enumerate(validation['clean_text']):
    chunks = simple_split(text)
    all_chunks.extend(chunks)
    chunk_pos.extend([i] * len(chunks))


In [14]:
embeddings = sentence_transf_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True
)

Batches: 100%|██████████| 497/497 [01:41<00:00,  4.91it/s]


In [15]:
def semantic_chunking(chunks, doc_ids, embeddings, window_size=2, t = 0.9, max_len=300):
    #0.9
    merged_chunks = []

    chunks = list(chunks)
    doc_ids = list(doc_ids)
    
    current_chunk = chunks[0]
    current_indices = [0]
    current_doc_id = doc_ids[0]
    
    for i in range(1, len(chunks)):
        
        if doc_ids[i] != current_doc_id:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
            continue
        
        # ventana
        window_indices = current_indices[-window_size:]
        window_embeddings = embeddings[window_indices]
        
        current_embedding = embeddings[i].reshape(1, -1)
        
        sim = cosine_similarity(current_embedding, window_embeddings).max()        
        if sim >= t and len(current_chunk.split()) + len(chunks[i].split()) <= max_len:
            current_chunk += " " + chunks[i]
            current_indices.append(i)
        else:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
    
    # último chunk
    merged_chunks.append({
        "text": current_chunk,
        "doc_id": current_doc_id,
        "chunk_indices": current_indices
    })
    
    return merged_chunks

In [16]:
merged = semantic_chunking(all_chunks, chunk_pos, embeddings)

In [17]:
merged_df = pd.DataFrame(merged)

In [18]:
merged_df_clean = merged_df[merged_df['text'].str.len() > 100]

In [19]:
merged_df_clean.head()

,text,doc_id,chunk_indices
0,Introduction Common sense understandings of pu...,0,[0]
1,literature by using Bacchi s 2009 What s the p...,0,[1]
2,the Council s written documents to further und...,0,[2]
3,"review Foucauldian Discourse Analysis, which f...",0,[3]
4,"of neoliberalism, Thatcherism welfare policy s...",0,[4]


In [26]:
validation = validation.reset_index(drop=True)

In [20]:
embeddings_chunks = sentence_transf_model.encode(
    merged_df_clean['text'].to_list(),
    batch_size=64,
    show_progress_bar=True
)

Batches: 100%|██████████| 398/398 [01:30<00:00,  4.41it/s]


In [27]:
summary_embedding = sentence_transf_model.encode(validation['summary'])[0]

merged_df_clean['scores'] = cosine_similarity(
    embeddings_chunks,
    [summary_embedding]
).flatten()

C:\Users\User\AppData\Local\Temp\ipykernel_24444\738744997.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_clean['scores'] = cosine_similarity(


In [28]:
merged_df_clean.head()

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,[0],0.932996
1,literature by using Bacchi s 2009 What s the p...,0,[1],0.914908
2,the Council s written documents to further und...,0,[2],0.848610
3,"review Foucauldian Discourse Analysis, which f...",0,[3],0.883236
4,"of neoliberalism, Thatcherism welfare policy s...",0,[4],0.812073


In [29]:
merged_df_clean['scores'].mean()

0.7633121

In [30]:
merged_df_clean['scores'].std()

0.033186425

In [ ]:
def top_pct(group, pct=0.3):
    return floor(len(group)*pct)

In [31]:
df_sorted = (
    merged_df_clean
    .sort_values(['doc_id','scores'], ascending=[True, False])
    .groupby('doc_id', group_keys=False)
    .head(6)
    #.apply(lambda x: x.head(max(1, int(len(x)*0.2))))
)

In [32]:
df_sorted

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,[0],0.932996
42,". As Gamble 1989 argues, the apparent contradi...",0,"[44, 45]",0.916704
1,literature by using Bacchi s 2009 What s the p...,0,[1],0.914908
57,political actors respond. Discourse as a const...,0,[61],0.891107
3,"review Foucauldian Discourse Analysis, which f...",0,[3],0.883236
...,...,...,...,...
25503,2 the role longleaf pine forests and forestry ...,299,[31750],0.811435
25532,study provide a deeper understanding of the fa...,299,"[31795, 31796]",0.797809
25480,has become an important policy and management ...,299,[31725],0.797531
25505,that emerged within an interview was assigned ...,299,[31752],0.793770


In [33]:
df_sorted['chunk_indices'] = df_sorted['chunk_indices'].apply(lambda x: min(x))

In [34]:
df_sorted.sort_values(['chunk_indices'], ascending=True, inplace=True)

In [35]:
df_sorted

,text,doc_id,chunk_indices,scores
0,Introduction Common sense understandings of pu...,0,0,0.932996
1,literature by using Bacchi s 2009 What s the p...,0,1,0.914908
3,"review Foucauldian Discourse Analysis, which f...",0,3,0.883236
40,"to reveal the what could have been, and engage...",0,42,0.876343
42,". As Gamble 1989 argues, the apparent contradi...",0,44,0.916704
...,...,...,...,...
25493,such as prescribed fire important to achieve r...,299,31738,0.819902
25498,on forest management decisions. Although there...,299,31745,0.785730
25503,2 the role longleaf pine forests and forestry ...,299,31750,0.811435
25505,that emerged within an interview was assigned ...,299,31752,0.793770


In [36]:
df_final = df_sorted.groupby('doc_id')['text'].apply(" ".join)

In [ ]:
df_final =pd.DataFrame(df_final)

In [46]:
df_final

,text
doc_id,
0,Introduction Common sense understandings of pu...
1,"in to participate in collaborative, inter- and..."
2,Introduction Swear and taboo words in the subt...
3,context While the WVS has been conducted in mo...
4,"Yet, the effects of time-averaging on archaeol..."
...,...
295,"on their support for CRBs. Within the survey, ..."
296,that cover the majority of workers within thei...
297,Introduction Cultural studies has often attemp...


In [47]:
df_final = df_final.join(train['summary'], how='left')

In [ ]:
df_final

,text,summary
doc_id,,
0,Introduction Common sense understandings of pu...,"In this article, Victor Fan argues that analys..."
1,"in to participate in collaborative, inter- and...",Problem definition: Physicians spend more than...
2,Introduction Swear and taboo words in the subt...,Literary translation is one of the most challe...
3,context While the WVS has been conducted in mo...,There is a long-running debate on evaluating f...
4,"Yet, the effects of time-averaging on archaeol...","Recently, ‘bimajyo’ (美魔女) came into focus in J..."
...,...,...
295,"on their support for CRBs. Within the survey, ...",Previous research has shown a consistent effec...
296,that cover the majority of workers within thei...,"In the hotel industry, the management of room ..."
297,Introduction Cultural studies has often attemp...,This study leverages sample data from the Amer...


In [49]:
df_final.to_pickle('validation_clean.pkl')

In [37]:
length = df_final.apply(len)

In [38]:
length.mean()

5241.73

In [42]:
length.std()

1975.7476728204738

In [31]:
np.percentile(merged_df_clean['scores'], [0, 25, 50, 75, 90, 100])

array([0.63136303, 0.7215758 , 0.74162096, 0.7636314 , 0.78436942,
       0.9272362 ])

In [34]:
top_k = int(len(merged_df_clean['scores']) * 0.3)
top_scores = sorted(merged_df_clean['scores'], reverse=True)[:top_k]

np.mean(top_scores)

0.7913983831250079

In [ ]:
k = int(len(chunks) * 0.3)

In [ ]:
validation['clean_chunks'] = validation['clean_chunks'].apply(cleaning_data)

In [ ]:
validation['clean_chunks'] = validation['clean_chunks'].apply(cleaning_data)

In [ ]:
def normalize(scores):
    scores = np.array(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

In [ ]:
def chunk_importance(chunks, summary):
    emb_chunks = sentence_transf_model.encode(chunks)
    emb_summary = sentence_transf_model.encode([summary])[0]

    similarities = cosine_similarity(emb_chunks, [emb_summary]).flatten()

    total = len(chunks)
    norm_sim = normalize(similarities) 
    scores_pos = [i/total for i in list(range(total))]
    norm_pos = normalize(scores_pos)
    
    return(norm_sim*0.8 + norm_pos*0.2)

In [ ]:
train['importance_score'] = validation['importance_score'].apply(chunk_importance)

In [22]:
sims = []

for i in range(1, len(embeddings_chunks)):
    sim = cosine_similarity(
        embeddings_chunks[i].reshape(1, -1),
        embeddings_chunks[i-1].reshape(1, -1)
    )[0][0]
    sims.append(sim)

print(min(sims), max(sims), sum(sims)/len(sims))

0.6417402 0.9787351 0.8680118135469157
